# YOUR FIRST LAB
## Your first Frontier LLM Project

By the end of this course, you will have built an autonomous Agentic AI solution with 7 agents that collaborate to solve a business problem. All in good time! We will start with something smaller...

Our goal is to code a new kind of Web Browser. Give it a URL, and it will respond with a summary. The Reader's Digest of the internet!!

Welcome to the wonderful world of Data Science experimentation! Simply click in each "cell" with code in it, such as the cell immediately below this text, and hit Shift+Return to execute that cell. Be sure to run every cell, starting at the top, in order.


### Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.


### Note: you'll need to set the Kernel with every notebook..

In [10]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to OpenAI (or Ollama)

The next cell is where we load in the environment variables in your `.env` file and connect to OpenAI.  

If you'd like to use free Ollama instead, please see the README section "Free Alternative to Paid APIs", and if you're not sure how to do this, there's a full solution in the solutions folder (day1_with_ollama.ipynb).

In [11]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


# Let's make a quick call to a Frontier model to get started, as a preview!

In [12]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [13]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.choices[0].message.content

'Hi there! Nice to meet you. Welcome to the chat.\n\nI’m ChatGPT, here to help with a wide range of things—answers and explanations, writing and editing, brainstorming, coding, planning, tutoring, translations, and more.\n\nA few quick ideas of what I can do:\n- Explain a concept or help with homework\n- Draft or edit emails, essays, resumes, or posts\n- Brainstorm ideas for stories, projects, gifts, or events\n- Help with basic coding, debugging, or learning a new language\n- Plan a trip, study schedule, or meal plan\n- Translate or paraphrase text\n\nWhat would you like to start with? Tell me a bit about your interests or a task you have in mind, and we’ll take it from there.'

## OK onwards with our first project

In [26]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)
print("\n\n\n")
eed = fetch_website_contents("https://freedium-mirror.cfd/https://python.plainenglish.io/creating-dynamic-presentations-with-python-e8e37d8bcdfa")
print(eed)


Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 400,000 enrolled across 190

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [16]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [17]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [18]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
response.choices[0].message.content

'2 + 2 equals 4.'

## And now let's build useful messages for GPT-4.1-mini, using a function

In [19]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [20]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n.

## Time to bring it together - the API for OpenAI is very simple!

In [21]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [22]:
summarize("https://edwarddonner.com")

"# Edward Donner's Website: Where AI Nerdom Meets Modest Bragging\n\nEd is your friendly neighborhood AI tinkerer, self-proclaimed LLM evangelist, and amateur electronic music producer (emphasis on *very* amateur). He’s the CTO of Nebula.io, doing the noble work of helping humans unlock their potential with AI, and also the ex-CEO of a startup that got snatched up in 2021 — because, why not?\n\nIf you’ve ever wanted an unsolicited, probably long-winded lecture about large language models, Ed's your guy. Thanks to his friends' patience running out, he channeled that into Udemy courses that somehow racked up 400,000 students worldwide. Talk about accidental fame.\n\nThe site also features:\n- An **AI Curriculum** for aspiring brainy coders\n- A Proficient AI Engineer path\n- Games like Connect Four and a fancy “Outsmart” LLM battle arena (the Hunger Games of chatbots)\n- A newsletter full of rare but valuable emails (he promises)\n\n**Latest news bites:**\n- Jan 4, 2026: AI Builder with 

In [24]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [25]:
#display_summary("https://edwarddonner.com")
display_summary("https://freedium-mirror.cfd/https://python.plainenglish.io/creating-dynamic-presentations-with-python-e8e37d8bcdfa")

Ah, behold the thrilling world of automating PowerPoint presentations with Python! Apparently, slapping together slides by hand is sooo 2010—why not let `python-pptx` do the boring parts for you? This site features a short, not-so-dry tutorial on how to install the library, create a presentation object, and add slides programmatically. 

Oh, and in case you care, the website just hit a milestone of 20GB stored data—because who doesn't love a good data hoarding announcement? No groundbreaking news, just a humble “thank you” for the growth and some Patreon/Ko-fi hints if you want to support their caffeine habit.

In summary: If you want to ditch the tedious gnashing of your teeth over PowerPoint and embrace code-driven slide wizardry, this is your stop. The rest? Just enthusiastic chatter about Python's versatility and a polite pat on the back for their data stash.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [19]:
display_summary("https://cnn.com")

# CNN: Your One-Stop Shop for News Overload

Welcome to CNN, where they cover *everything* from global conflicts (Ukraine-Russia, Israel-Hamas) to the latest in celebrity gossip, sports, science breakthroughs, and even your weather forecast—because who doesn’t need a side of climate crisis with their morning coffee?

### Breaking News & Announcements
- The site is *brimming* with up-to-the-minute news and live TV streams.
- They’re super eager for feedback, especially about ads. Expect pop-ups asking if ads annoyed you, froze your screen, or made the cat run away.
- With multiple editions (US, International, Arabic, Español), CNN is basically everywhere, like that clingy friend who follows you around but with more serious headlines.

### Categories Galore
Politics, business, tech, entertainment, health, style, travel, and yes, even a section dedicated to "Life, But Better" (because you obviously need it).

### Summary
If you want news bites, in every flavor imaginable, and a smorgasbord of videos, polls, and live updates, CNN is your chaotic, caffeinated fix. Just brace yourself for the relentless ad feedback requests—they really *care* if that video ad was "too loud."

In [20]:
display_summary("https://anthropic.com")

# Anthropic: AI with a Conscience

Welcome to Anthropic, where AI isn't just about flashy tech but also not blowing up the world. They’re a public benefit corporation swimming in high-minded ideals like safety, transparency, and responsible scaling. Their crown jewel? Claude Opus 4.5 — apparently the best model for coding, agents, and enterprise workflows. (Because who doesn’t want their AI to be both a genius coder and the office MVP?)

Newsflash: They just dropped Claude Opus 4.5, boasting advanced tool use on their developer platform. It’s like giving your AI superpowers but keeping it on a leash.

In summary: Anthropic builds AI that aims not to doom humanity but maybe just to help it flourish, all while juggling the thrills and chills of cutting-edge AI research. Safe, smart, and slightly earnest.

I need to add a part related to the setup
add explanation to the message and roles

Summary of the code

In [22]:
# Step 1: Create your prompts

system_prompt = "something here"
user_prompt = """
    Lots of text
    Can be pasted here
"""

# Step 2: Make the messages list

messages = [] # fill this in

# Step 3: Call OpenAI
# response =

# Step 4: print the result
# print(